# 🎨 Text-to-Image Generator (Pretrained Stable Diffusion)

A simple project that uses a **pretrained diffusion model** (Stable Diffusion v1.5 from Hugging Face) to generate images from text prompts, wrapped in an interactive **Gradio** web app.

**How to run in Colab:**
1. `Runtime > Change runtime type > GPU (T4)`
2. Run every cell top to bottom
3. Click the public Gradio link that appears at the bottom

No training required — this uses a model that's already been trained on billions of image-text pairs. You're building an *application* on top of it, which is exactly what most real-world AI products do.

## 1. Install dependencies

In [1]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers accelerate gradio safetensors

## 2. Load the pretrained model
We're using `runwayml/stable-diffusion-v1-5`, a widely used open pretrained checkpoint. `torch_dtype=torch.float16` keeps it fast and light enough for a free Colab GPU.

In [2]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

MODEL_ID = "runwayml/stable-diffusion-v1-5"

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

torch_dtype = torch.float16 if device in {"cuda", "mps"} else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    safety_checker=None,  # remove for faster loading; add back for public deployment
)

# Faster, higher-quality scheduler than the default
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

if device == "cuda":
    pipe.enable_attention_slicing()  # reduces memory usage

print("Model loaded successfully!")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Using device: cuda


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Model loaded successfully!


## 3. Define the generation function

In [3]:
import random

def generate_image(prompt, negative_prompt, num_steps, guidance_scale, width, height, seed):
    if seed == -1 or seed is None:
        seed = random.randint(0, 2**32 - 1)
    generator = torch.Generator(device=device).manual_seed(int(seed))

    with torch.autocast(device) if device == "cuda" else torch.no_grad():
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt if negative_prompt else None,
            num_inference_steps=int(num_steps),
            guidance_scale=float(guidance_scale),
            width=int(width),
            height=int(height),
            generator=generator,
        )
    image = result.images[0]
    return image, seed

## 4. Build the Gradio interface

In [4]:
import gradio as gr

EXAMPLES = [
    ["a serene Japanese garden at sunrise, watercolor style", "blurry, low quality", 30, 7.5, 512, 512, -1],
    ["a futuristic city skyline at night, neon lights, cyberpunk", "blurry, low quality, distorted", 30, 7.5, 512, 512, -1],
    ["a cute robot reading a book in a cozy library, digital art", "blurry, low quality", 30, 7.5, 512, 512, -1],
]

with gr.Blocks(title="Text-to-Image Generator") as demo:
    gr.Markdown("# 🎨 Text-to-Image Generator\nPowered by a pretrained Stable Diffusion model.")

    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", placeholder="a photo of an astronaut riding a horse on mars", lines=2)
            negative_prompt = gr.Textbox(label="Negative prompt (what to avoid)", placeholder="blurry, low quality, distorted", lines=1)

            with gr.Accordion("Advanced settings", open=False):
                num_steps = gr.Slider(10, 50, value=30, step=1, label="Inference steps (more = higher quality, slower)")
                guidance_scale = gr.Slider(1, 15, value=7.5, step=0.5, label="Guidance scale (how closely to follow the prompt)")
                width = gr.Dropdown([384, 512, 640, 768], value=512, label="Width")
                height = gr.Dropdown([384, 512, 640, 768], value=512, label="Height")
                seed = gr.Number(value=-1, label="Seed (-1 = random)")

            generate_btn = gr.Button("Generate Image", variant="primary")

        with gr.Column(scale=1):
            output_image = gr.Image(label="Generated Image", type="pil")
            used_seed = gr.Number(label="Seed used", interactive=False)

    gr.Examples(
        examples=EXAMPLES,
        inputs=[prompt, negative_prompt, num_steps, guidance_scale, width, height, seed],
    )

    generate_btn.click(
        fn=generate_image,
        inputs=[prompt, negative_prompt, num_steps, guidance_scale, width, height, seed],
        outputs=[output_image, used_seed],
    )

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2de96ed8b728d942d4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Next steps / ways to extend this project
- Swap `MODEL_ID` for another pretrained checkpoint (e.g. `stabilityai/stable-diffusion-2-1`, `stabilityai/sdxl-turbo` for speed, or a fine-tuned art-style model from Hugging Face Hub)
- Add image-to-image generation (`StableDiffusionImg2ImgPipeline`) so users can upload a sketch and get a rendered version
- Add an upscaler (`stabilityai/stable-diffusion-x4-upscaler`) as a post-processing step
- Deploy permanently on Hugging Face Spaces instead of a temporary Colab link
- Add a gallery/history panel that stores previously generated images in the session